# FER-2013 Enhanced Emotion Detection Model Training

This notebook trains a CNN model using the FER-2013 enhanced dataset for real-time emotion detection.

## Dataset Information
- **Dataset**: FER-2013 Enhanced
- **Emotions**: 7 classes (angry, disgust, fear, happy, neutral, sad, surprise)
- **Image Size**: 48x48 grayscale
- **Total Records**: 3,501 samples

## Model Architecture
- Convolutional Neural Network (CNN)
- Multiple Conv2D layers with BatchNormalization
- Dropout for regularization
- Adam optimizer with learning rate scheduling

## 1. Import Required Libraries

In [ ]:
# FER-2013 Enhanced Emotion Detection Model Training
# Train a CNN model using the FER-2013 enhanced dataset for real-time emotion detection

import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import pandas as pd

# Computer vision
try:
    import cv2
    print("✅ OpenCV imported successfully")
except ImportError:
    print("⚠️ OpenCV not available, using PIL instead")
    cv2 = None

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
import pickle
import json
from datetime import datetime
import logging

# Configure matplotlib for inline display
plt.style.use('default')
%matplotlib inline

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

print("✅ Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Available: {len(gpus)} device(s)")
    for gpu in gpus:
        print(f"   - {gpu}")
else:
    print("⚠️ No GPU detected, using CPU")

# Set memory growth for GPU (if available)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU memory growth configured")
    except RuntimeError as e:
        print(f"⚠️ GPU configuration warning: {e}")

## 2. Define FER2013 Emotion Trainer Class

In [ ]:
class FER2013EmotionTrainer:
    """FER-2013 Enhanced Dataset Emotion Model Trainer"""
    
    def __init__(self):
        self.emotions = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
        self.emotion_mapping = {emotion: idx for idx, emotion in enumerate(self.emotions)}
        self.num_classes = len(self.emotions)
        self.img_size = 48
        self.model = None
        self.history = None
        
        print(f"✅ FER2013 Trainer initialized")
        print(f"Emotions: {self.emotions}")
        print(f"Number of classes: {self.num_classes}")
        print(f"Image size: {self.img_size}x{self.img_size}")
    
    def load_fer2013_data(self):
        """Load FER-2013 enhanced dataset"""
        print("📊 Loading FER-2013 Enhanced Dataset...")
        
        # Dataset file paths to try
        dataset_paths = [
            'emotion_datasets/fer2013/fer2013_enhanced.csv',
            '../emotion_datasets/fer2013/fer2013_enhanced.csv',
            'fer2013_enhanced.csv'
        ]
        
        df = None
        for dataset_path in dataset_paths:
            if os.path.exists(dataset_path):
                try:
                    df = pd.read_csv(dataset_path)
                    print(f"✅ Loaded {len(df)} samples from {dataset_path}")
                    break
                except Exception as e:
                    print(f"⚠️ Error loading {dataset_path}: {e}")
                    continue
        
        if df is None:
            print("❌ Dataset not found. Creating sample data for demonstration...")
            df = self.create_sample_data()
        
        # Validate dataset structure
        required_columns = ['emotion', 'pixels']
        if not all(col in df.columns for col in required_columns):
            raise ValueError(f"Dataset must contain columns: {required_columns}")
        
        # Display dataset info
        print("\n📈 Dataset Statistics:")
        print(df['emotion'].value_counts())
        
        return df
    
    def create_sample_data(self):
        """Create sample data for demonstration if dataset not found"""
        print("🔧 Creating sample dataset for demonstration...")
        
        sample_data = []
        samples_per_emotion = 50
        
        for emotion in self.emotions:
            for i in range(samples_per_emotion):
                # Create random 48x48 pixel data
                pixels = np.random.randint(0, 256, self.img_size * self.img_size)
                pixel_string = ' '.join(map(str, pixels))
                
                sample_data.append({
                    'emotion': emotion,
                    'pixels': pixel_string
                })
        
        df = pd.DataFrame(sample_data)
        print(f"✅ Created sample dataset with {len(df)} samples")
        return df
    
    def preprocess_data(self, df):
        """Preprocess the dataset"""
        print("🔧 Preprocessing data...")
        
        # Extract pixel data and labels
        pixels = []
        labels = []
        
        for idx, row in df.iterrows():
            try:
                # Convert pixel string to array
                pixel_values = [int(pixel) for pixel in str(row['pixels']).split()]
                
                # Ensure we have the right number of pixels
                if len(pixel_values) != self.img_size * self.img_size:
                    print(f"⚠️ Skipping row {idx}: expected {self.img_size * self.img_size} pixels, got {len(pixel_values)}")
                    continue
                
                pixel_array = np.array(pixel_values).reshape(self.img_size, self.img_size)
                
                pixels.append(pixel_array)
                labels.append(row['emotion'])
                
            except Exception as e:
                print(f"⚠️ Error processing row {idx}: {e}")
                continue
        
        if len(pixels) == 0:
            raise ValueError("No valid data found after preprocessing")
        
        # Convert to numpy arrays
        X = np.array(pixels, dtype='float32')
        y = np.array([self.emotion_mapping.get(emotion, 0) for emotion in labels])
        
        # Normalize pixel values
        X = X / 255.0
        
        # Reshape for CNN (add channel dimension)
        X = X.reshape(-1, self.img_size, self.img_size, 1)
        
        # Convert labels to categorical
        y = to_categorical(y, self.num_classes)
        
        print(f"✅ Data preprocessed: X shape {X.shape}, y shape {y.shape}")
        print(f"   X data type: {X.dtype}, range: [{X.min():.3f}, {X.max():.3f}]")
        
        return X, y
    
    def create_model(self):
        """Create CNN model architecture"""
        print("🏗️ Creating CNN model architecture...")
        
        try:
            model = Sequential([
                # First Convolutional Block
                Conv2D(32, (3, 3), activation='relu', input_shape=(self.img_size, self.img_size, 1), name='conv1_1'),
                BatchNormalization(name='bn1_1'),
                Conv2D(32, (3, 3), activation='relu', name='conv1_2'),
                MaxPooling2D(pool_size=(2, 2), name='pool1'),
                Dropout(0.25, name='dropout1'),
                
                # Second Convolutional Block
                Conv2D(64, (3, 3), activation='relu', name='conv2_1'),
                BatchNormalization(name='bn2_1'),
                Conv2D(64, (3, 3), activation='relu', name='conv2_2'),
                MaxPooling2D(pool_size=(2, 2), name='pool2'),
                Dropout(0.25, name='dropout2'),
                
                # Third Convolutional Block
                Conv2D(128, (3, 3), activation='relu', name='conv3_1'),
                BatchNormalization(name='bn3_1'),
                Dropout(0.25, name='dropout3'),
                
                # Fully Connected Layers
                Flatten(name='flatten'),
                Dense(512, activation='relu', name='dense1'),
                BatchNormalization(name='bn_dense1'),
                Dropout(0.5, name='dropout4'),
                Dense(256, activation='relu', name='dense2'),
                Dropout(0.5, name='dropout5'),
                Dense(self.num_classes, activation='softmax', name='output')
            ])
            
            # Compile model with error handling
            try:
                optimizer = Adam(learning_rate=0.001)
            except TypeError:
                # For older TensorFlow versions
                optimizer = Adam(lr=0.001)
            
            model.compile(
                optimizer=optimizer,
                loss='categorical_crossentropy',
                metrics=['accuracy']
            )
            
            self.model = model
            
            print("✅ Model created successfully!")
            return model
            
        except Exception as e:
            print(f"❌ Error creating model: {e}")
            raise

# Initialize trainer with error handling
try:
    trainer = FER2013EmotionTrainer()
    print("\n🎯 Trainer ready for use!")
except Exception as e:
    print(f"❌ Error initializing trainer: {e}")
    trainer = None

## 3. Load and Explore Dataset

In [ ]:
# Check if trainer was initialized successfully
if trainer is None:
    print("❌ Trainer not initialized. Please run the previous cell first.")
else:
    try:
        # Load the FER-2013 enhanced dataset
        df = trainer.load_fer2013_data()
        
        # Display first few rows
        print("\n📋 First 5 rows of dataset:")
        print(df.head())
        
        # Show dataset shape
        print(f"\n📊 Dataset shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        
        # Check for missing values
        print(f"\n🔍 Missing values:")
        print(df.isnull().sum())
        
        # Show data types
        print(f"\n📋 Data types:")
        print(df.dtypes)
        
    except Exception as e:
        print(f"❌ Error loading dataset: {e}")
        df = None

## 4. Visualize Sample Images

In [ ]:
# Visualize sample images from each emotion class
if trainer is not None and df is not None:
    try:
        plt.figure(figsize=(15, 10))
        
        emotions = trainer.emotions
        samples_per_emotion = 3
        
        plot_count = 0
        
        for i, emotion in enumerate(emotions):
            # Get samples for this emotion
            emotion_samples = df[df['emotion'] == emotion].head(samples_per_emotion)
            
            if len(emotion_samples) == 0:
                print(f"⚠️ No samples found for emotion: {emotion}")
                continue
            
            for j, (idx, row) in enumerate(emotion_samples.iterrows()):
                try:
                    # Convert pixel string to image
                    pixel_values = [int(pixel) for pixel in str(row['pixels']).split()]
                    
                    if len(pixel_values) != 48 * 48:
                        print(f"⚠️ Skipping invalid pixel data for {emotion} sample {j}")
                        continue
                    
                    img = np.array(pixel_values).reshape(48, 48)
                    
                    # Plot image
                    plot_count += 1
                    plt.subplot(len(emotions), samples_per_emotion, plot_count)
                    plt.imshow(img, cmap='gray')
                    plt.title(f'{emotion.upper()}', fontsize=10)
                    plt.axis('off')
                    
                except Exception as e:
                    print(f"⚠️ Error plotting {emotion} sample {j}: {e}")
                    continue
        
        plt.tight_layout()
        plt.suptitle('Sample Images from FER-2013 Enhanced Dataset', y=1.02, fontsize=16)
        plt.show()
        
        print("✅ Sample images displayed!")
        
    except Exception as e:
        print(f"❌ Error visualizing samples: {e}")
        print("Creating simple visualization instead...")
        
        # Simple fallback visualization
        plt.figure(figsize=(10, 6))
        emotion_counts = df['emotion'].value_counts()
        plt.bar(emotion_counts.index, emotion_counts.values)
        plt.title('Emotion Distribution in Dataset')
        plt.xlabel('Emotion')
        plt.ylabel('Count')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        
else:
    print("❌ Cannot visualize: trainer or dataset not available")

## 5. Preprocess Data

In [ ]:
# Preprocess the data
if trainer is not None and df is not None:
    try:
        X, y = trainer.preprocess_data(df)
        
        print(f"\n📊 Preprocessed Data:")
        print(f"X shape: {X.shape}")
        print(f"y shape: {y.shape}")
        print(f"X data type: {X.dtype}")
        print(f"X value range: [{X.min():.3f}, {X.max():.3f}]")
        
        # Check for any NaN or infinite values
        if np.isnan(X).any():
            print("⚠️ Warning: NaN values found in X")
            X = np.nan_to_num(X, nan=0.0)
            
        if np.isinf(X).any():
            print("⚠️ Warning: Infinite values found in X")
            X = np.nan_to_num(X, posinf=1.0, neginf=0.0)
        
        # Split data into train, validation, and test sets with error handling
        try:
            # Check if we have enough samples for stratification
            min_samples = np.min(np.sum(y, axis=0))
            if min_samples < 2:
                print("⚠️ Warning: Some classes have very few samples. Using random split instead of stratified.")
                X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
                X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
            else:
                # Convert back to class indices for stratification
                y_indices = np.argmax(y, axis=1)
                X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y_indices)
                
                # For second split, get indices again
                y_temp_indices = np.argmax(y_temp, axis=1)
                X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp_indices)
            
            print(f"\n📊 Data Split:")
            print(f"Training set: {X_train.shape[0]} samples")
            print(f"Validation set: {X_val.shape[0]} samples")
            print(f"Test set: {X_test.shape[0]} samples")
            
            # Verify splits
            total_samples = X_train.shape[0] + X_val.shape[0] + X_test.shape[0]
            print(f"Total samples: {total_samples} (original: {X.shape[0]})")
            
        except Exception as e:
            print(f"❌ Error splitting data: {e}")
            print("Using simple random split...")
            
            # Simple random split as fallback
            n_samples = X.shape[0]
            n_train = int(0.7 * n_samples)
            n_val = int(0.15 * n_samples)
            
            indices = np.random.permutation(n_samples)
            train_idx = indices[:n_train]
            val_idx = indices[n_train:n_train+n_val]
            test_idx = indices[n_train+n_val:]
            
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]
            X_test, y_test = X[test_idx], y[test_idx]
            
            print(f"\n📊 Data Split (random):")
            print(f"Training set: {X_train.shape[0]} samples")
            print(f"Validation set: {X_val.shape[0]} samples")
            print(f"Test set: {X_test.shape[0]} samples")
        
    except Exception as e:
        print(f"❌ Error preprocessing data: {e}")
        X, y = None, None
        X_train = X_val = X_test = None
        y_train = y_val = y_test = None
        
else:
    print("❌ Cannot preprocess: trainer or dataset not available")
    X, y = None, None
    X_train = X_val = X_test = None
    y_train = y_val = y_test = None

## 6. Create and Display Model Architecture

In [ ]:
# Create the model
if trainer is not None and X_train is not None:
    try:
        model = trainer.create_model()
        
        # Display model summary
        print("\n🏗️ Model Architecture:")
        model.summary()
        
        # Count total parameters
        total_params = model.count_params()
        print(f"\n📊 Total Parameters: {total_params:,}")
        
        # Calculate model size estimate
        model_size_mb = (total_params * 4) / (1024 * 1024)  # Assuming float32
        print(f"📊 Estimated Model Size: {model_size_mb:.2f} MB")
        
    except Exception as e:
        print(f"❌ Error creating model: {e}")
        model = None
        total_params = 0
        
else:
    print("❌ Cannot create model: trainer or training data not available")
    model = None
    total_params = 0

## 7. Set Up Training Callbacks

In [ ]:
# Set up training callbacks
if model is not None:
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = f"fer2013_emotion_model_{timestamp}"
        
        # Create callbacks with error handling
        callbacks = []
        
        # Early Stopping
        try:
            early_stopping = EarlyStopping(
                monitor='val_accuracy',
                patience=10,
                restore_best_weights=True,
                verbose=1
            )
            callbacks.append(early_stopping)
            print("✅ Early Stopping callback added")
        except Exception as e:
            print(f"⚠️ Could not add Early Stopping: {e}")
        
        # Model Checkpoint
        try:
            checkpoint = ModelCheckpoint(
                filepath=f'{model_name}_best.h5',
                monitor='val_accuracy',
                save_best_only=True,
                verbose=1
            )
            callbacks.append(checkpoint)
            print("✅ Model Checkpoint callback added")
        except Exception as e:
            print(f"⚠️ Could not add Model Checkpoint: {e}")
        
        # Learning Rate Reduction
        try:
            lr_reducer = ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=0.0001,
                verbose=1
            )
            callbacks.append(lr_reducer)
            print("✅ Learning Rate Reduction callback added")
        except Exception as e:
            print(f"⚠️ Could not add Learning Rate Reduction: {e}")
        
        print(f"\n✅ Training callbacks configured ({len(callbacks)} callbacks):")
        if len(callbacks) > 0:
            print("- Early Stopping (patience=10)" if any(isinstance(cb, EarlyStopping) for cb in callbacks) else "")
            print("- Model Checkpoint (save best model)" if any(isinstance(cb, ModelCheckpoint) for cb in callbacks) else "")
            print("- Learning Rate Reduction (factor=0.5, patience=5)" if any(isinstance(cb, ReduceLROnPlateau) for cb in callbacks) else "")
            print(f"- Model will be saved as: {model_name}_best.h5")
        else:
            print("⚠️ No callbacks configured - training will proceed without callbacks")
        
    except Exception as e:
        print(f"❌ Error setting up callbacks: {e}")
        callbacks = []
        model_name = f"fer2013_emotion_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
else:
    print("❌ Cannot set up callbacks: model not available")
    callbacks = []
    model_name = f"fer2013_emotion_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

## 8. Train the Model

In [ ]:
# Train the model
if model is not None and X_train is not None and y_train is not None:
    try:
        print("🚀 Starting model training...")
        print(f"Training samples: {len(X_train)}")
        print(f"Validation samples: {len(X_val) if X_val is not None else 0}")
        print(f"Epochs: 50")
        print(f"Batch size: 32")
        
        # Determine validation data
        validation_data = (X_val, y_val) if X_val is not None and y_val is not None else None
        
        if validation_data is None:
            print("⚠️ No validation data available - using training data for validation")
            validation_split = 0.2
            validation_data = None
        else:
            validation_split = 0.0
        
        # Start training with error handling
        history = model.fit(
            X_train, y_train,
            batch_size=32,
            epochs=50,
            validation_data=validation_data,
            validation_split=validation_split,
            callbacks=callbacks,
            verbose=1
        )
        
        trainer.history = history
        print("\n✅ Training completed!")
        
    except KeyboardInterrupt:
        print("\n⚠️ Training interrupted by user")
        history = None
        
    except Exception as e:
        print(f"❌ Error during training: {e}")
        print("Attempting simplified training...")
        
        try:
            # Simplified training without callbacks
            history = model.fit(
                X_train, y_train,
                batch_size=32,
                epochs=10,  # Reduced epochs
                validation_split=0.2,
                verbose=1
            )
            
            trainer.history = history
            print("\n✅ Simplified training completed!")
            
        except Exception as e2:
            print(f"❌ Simplified training also failed: {e2}")
            history = None
            
else:
    print("❌ Cannot train: model or training data not available")
    history = None

## 9. Visualize Training Progress

In [ ]:
# Plot training history
if history is not None:
    try:
        plt.figure(figsize=(15, 5))
        
        # Check what metrics are available
        available_metrics = list(history.history.keys())
        print(f"Available metrics: {available_metrics}")
        
        # Plot training & validation accuracy
        plt.subplot(1, 2, 1)
        if 'accuracy' in history.history:
            plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
        if 'val_accuracy' in history.history:
            plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
        elif 'accuracy' in history.history:
            # If no validation accuracy, just show training
            plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
        
        plt.title('Model Accuracy', fontsize=14)
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Plot training & validation loss
        plt.subplot(1, 2, 2)
        if 'loss' in history.history:
            plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
        if 'val_loss' in history.history:
            plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
        elif 'loss' in history.history:
            # If no validation loss, just show training
            plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
        
        plt.title('Model Loss', fontsize=14)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print final metrics
        try:
            if 'accuracy' in history.history and len(history.history['accuracy']) > 0:
                final_train_acc = history.history['accuracy'][-1]
                print(f"Final Training Accuracy: {final_train_acc:.4f}")
            
            if 'val_accuracy' in history.history and len(history.history['val_accuracy']) > 0:
                final_val_acc = history.history['val_accuracy'][-1]
                print(f"Final Validation Accuracy: {final_val_acc:.4f}")
            
            if 'loss' in history.history and len(history.history['loss']) > 0:
                final_train_loss = history.history['loss'][-1]
                print(f"Final Training Loss: {final_train_loss:.4f}")
            
            if 'val_loss' in history.history and len(history.history['val_loss']) > 0:
                final_val_loss = history.history['val_loss'][-1]
                print(f"Final Validation Loss: {final_val_loss:.4f}")
                
        except Exception as e:
            print(f"⚠️ Could not extract final metrics: {e}")
        
    except Exception as e:
        print(f"❌ Error plotting training history: {e}")
        print("Training completed but visualization failed")
        
else:
    print("❌ No training history available to plot")
    # Set default values for later use
    final_train_acc = 0.0
    final_val_acc = 0.0
    final_train_loss = 0.0
    final_val_loss = 0.0

## 10. Evaluate Model on Test Set

In [ ]:
# Evaluate on test set
if model is not None and X_test is not None and y_test is not None:
    try:
        print("🧪 Evaluating model on test set...")
        
        test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
        print(f"\n📊 Test Results:")
        print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
        print(f"Test Loss: {test_loss:.4f}")
        
        # Make predictions on test set
        print("\n🔮 Making predictions on test set...")
        y_pred = model.predict(X_test, verbose=0)
        y_pred_classes = np.argmax(y_pred, axis=1)
        y_true_classes = np.argmax(y_test, axis=1)
        
        # Classification report
        print("\n📋 Classification Report:")
        try:
            report = classification_report(y_true_classes, y_pred_classes, target_names=trainer.emotions)
            print(report)
        except Exception as e:
            print(f"⚠️ Could not generate classification report: {e}")
            
            # Simple accuracy calculation as fallback
            accuracy = np.mean(y_pred_classes == y_true_classes)
            print(f"Simple accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
        
    except Exception as e:
        print(f"❌ Error evaluating model: {e}")
        test_accuracy = 0.0
        test_loss = 0.0
        y_pred_classes = None
        y_true_classes = None
        
else:
    print("❌ Cannot evaluate: model or test data not available")
    test_accuracy = 0.0
    test_loss = 0.0
    y_pred_classes = None
    y_true_classes = None

## 11. Create Confusion Matrix

In [ ]:
# Create confusion matrix
if y_pred_classes is not None and y_true_classes is not None and trainer is not None:
    try:
        cm = confusion_matrix(y_true_classes, y_pred_classes)
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=trainer.emotions, 
                    yticklabels=trainer.emotions,
                    cbar_kws={'label': 'Count'})
        plt.title('Confusion Matrix - FER2013 Emotion Detection', fontsize=16)
        plt.xlabel('Predicted Emotion', fontsize=12)
        plt.ylabel('True Emotion', fontsize=12)
        plt.xticks(rotation=45)
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()
        
        # Calculate per-class accuracy
        print("\n📊 Per-Class Accuracy:")
        for i, emotion in enumerate(trainer.emotions):
            if cm.shape[0] > i and cm.shape[1] > i:
                class_accuracy = cm[i, i] / cm[i, :].sum() if cm[i, :].sum() > 0 else 0
                print(f"{emotion.capitalize()}: {class_accuracy:.4f} ({class_accuracy*100:.2f}%)")
            else:
                print(f"{emotion.capitalize()}: No samples in test set")
        
        # Overall statistics
        total_correct = np.trace(cm)
        total_samples = np.sum(cm)
        overall_accuracy = total_correct / total_samples if total_samples > 0 else 0
        print(f"\n📊 Overall Test Accuracy: {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)")
        
    except Exception as e:
        print(f"❌ Error creating confusion matrix: {e}")
        
        # Simple accuracy plot as fallback
        try:
            if y_pred_classes is not None and y_true_classes is not None:
                accuracy = np.mean(y_pred_classes == y_true_classes)
                
                plt.figure(figsize=(8, 6))
                plt.bar(['Correct', 'Incorrect'], 
                       [accuracy * 100, (1 - accuracy) * 100],
                       color=['green', 'red'], alpha=0.7)
                plt.title('Model Performance on Test Set')
                plt.ylabel('Percentage')
                plt.ylim(0, 100)
                
                for i, v in enumerate([accuracy * 100, (1 - accuracy) * 100]):
                    plt.text(i, v + 1, f'{v:.1f}%', ha='center', va='bottom')
                
                plt.tight_layout()
                plt.show()
                
        except Exception as e2:
            print(f"⚠️ Fallback visualization also failed: {e2}")
        
else:
    print("❌ Cannot create confusion matrix: predictions or trainer not available")

## 12. Save Model and Metadata

In [ ]:
# Save final model and metadata
if model is not None and model_name is not None:
    try:
        # Save final model
        final_model_path = f'{model_name}_final.h5'
        model.save(final_model_path)
        print(f"✅ Final model saved: {final_model_path}")
        
        # Prepare metadata with safe defaults
        metadata = {
            'model_name': model_name,
            'dataset': 'FER2013-Enhanced',
            'total_samples': len(df) if df is not None and df is not None else 0,
            'train_samples': len(X_train) if X_train is not None and X_train is not None else 0,
            'val_samples': len(X_val) if 'X_val' in locals() and X_val is not None else 0,
            'test_samples': len(X_test) if 'X_test' in locals() and X_test is not None else 0,
            'emotions': trainer.emotions if trainer is not None else [],
            'num_classes': trainer.num_classes if trainer is not None else 0,
            'img_size': trainer.img_size if trainer is not None else 48,
            'total_parameters': total_params if total_params is not None else 0,
            'training_time': datetime.now().strftime("%Y%m%d_%H%M%S"),
            'tensorflow_version': tf.__version__
        }
        
        # Add training metrics if available
        if final_train_acc is not None:
            metadata['final_train_accuracy'] = float(final_train_acc)
        if final_val_acc is not None:
            metadata['final_val_accuracy'] = float(final_val_acc)
        if test_accuracy is not None:
            metadata['test_accuracy'] = float(test_accuracy)
        if test_loss is not None:
            metadata['test_loss'] = float(test_loss)
        if history is not None:
            metadata['epochs_trained'] = len(history.history.get('accuracy', []))
        
        # Save metadata
        metadata_path = f'{model_name}_metadata.json'
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)
        print(f"✅ Metadata saved: {metadata_path}")
        
        # Save emotion mapping if trainer available
        if trainer is not None:
            mapping_path = f'{model_name}_emotion_mapping.pkl'
            with open(mapping_path, 'wb') as f:
                pickle.dump(trainer.emotion_mapping, f)
            print(f"✅ Emotion mapping saved: {mapping_path}")
        
        # Save training history if available
        if history is not None:
            history_path = f'{model_name}_history.pkl'
            with open(history_path, 'wb') as f:
                pickle.dump(history.history, f)
            print(f"✅ Training history saved: {history_path}")
        
    except Exception as e:
        print(f"❌ Error saving model/metadata: {e}")
        print("Model training completed but saving failed")
        
        # Try to save just the model
        try:
            simple_model_path = f"fer2013_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}.h5"
            model.save(simple_model_path)
            print(f"✅ Model saved with simple name: {simple_model_path}")
        except Exception as e2:
            print(f"❌ Could not save model at all: {e2}")
        
else:
    print("❌ Cannot save: model not available")

## 13. Test Model with Sample Predictions

In [ ]:
# Test model with sample predictions
if model is not None and X_test is not None and y_test is not None and trainer is not None:
    try:
        print("🧪 Testing model with sample predictions...")
        
        # Select random test samples
        num_samples = min(6, len(X_test))  # Ensure we don't exceed available samples
        random_indices = np.random.choice(len(X_test), num_samples, replace=False)
        
        plt.figure(figsize=(15, 8))
        
        for i, idx in enumerate(random_indices):
            try:
                # Get sample
                sample_image = X_test[idx]
                true_label = np.argmax(y_test[idx])
                
                # Make prediction
                prediction = model.predict(sample_image.reshape(1, 48, 48, 1), verbose=0)
                predicted_label = np.argmax(prediction)
                confidence = prediction[0][predicted_label]
                
                # Plot image
                plt.subplot(2, 3, i + 1)
                plt.imshow(sample_image.reshape(48, 48), cmap='gray')
                
                # Set title with prediction results
                if true_label < len(trainer.emotions) and predicted_label < len(trainer.emotions):
                    true_emotion = trainer.emotions[true_label]
                    pred_emotion = trainer.emotions[predicted_label]
                    
                    if true_label == predicted_label:
                        title_color = 'green'
                        status = '✅'
                    else:
                        title_color = 'red'
                        status = '❌'
                    
                    plt.title(f'{status} True: {true_emotion}\nPred: {pred_emotion} ({confidence:.2f})', 
                              color=title_color, fontsize=10)
                else:
                    plt.title(f'Label: {true_label}\nPred: {predicted_label} ({confidence:.2f})', 
                              fontsize=10)
                
                plt.axis('off')
                
            except Exception as e:
                print(f"⚠️ Error processing sample {i}: {e}")
                continue
        
        plt.tight_layout()
        plt.suptitle('Sample Predictions on Test Set', y=1.02, fontsize=16)
        plt.show()
        
        print("✅ Sample predictions completed!")
        
    except Exception as e:
        print(f"❌ Error testing sample predictions: {e}")
        
        # Simple fallback test
        try:
            if len(X_test) > 0:
                sample = X_test[0:1]  # Take first sample
                pred = model.predict(sample, verbose=0)
                pred_class = np.argmax(pred)
                confidence = pred[0][pred_class]
                
                print(f"✅ Simple test successful:")
                print(f"   Predicted class: {pred_class}")
                print(f"   Confidence: {confidence:.4f}")
                if trainer is not None and pred_class < len(trainer.emotions):
                    print(f"   Emotion: {trainer.emotions[pred_class]}")
                    
        except Exception as e2:
            print(f"⚠️ Simple test also failed: {e2}")
        
else:
    print("❌ Cannot test predictions: model, test data, or trainer not available")

## 14. Model Summary and Next Steps

In [ ]:
# Final summary
print("\n🎯 FER2013 Emotion Detection Model Training Complete!")
print("="*60)

# Model Performance Summary
print(f"📊 Model Performance:")
if test_accuracy is not None and test_accuracy > 0:
    print(f"   • Test Accuracy: {test_accuracy*100:.2f}%")
else:
    print(f"   • Test Accuracy: Not available")

if total_params is not None and total_params > 0:
    print(f"   • Total Parameters: {total_params:,}")
else:
    print(f"   • Total Parameters: Not available")

if X_train is not None and X_train is not None:
    print(f"   • Training Samples: {len(X_train):,}")
else:
    print(f"   • Training Samples: Not available")

if trainer is not None:
    print(f"   • Emotions Detected: {len(trainer.emotions)}")
    print(f"   • Emotion Classes: {', '.join(trainer.emotions)}")
else:
    print(f"   • Emotions Detected: Not available")

# Generated Files
print(f"\n📁 Generated Files:")
if model_name is not None:
    print(f"   • Best Model: {model_name}_best.h5 (if training completed)")
    print(f"   • Final Model: {model_name}_final.h5")
    print(f"   • Metadata: {model_name}_metadata.json")
    print(f"   • Emotion Mapping: {model_name}_emotion_mapping.pkl")
    print(f"   • Training History: {model_name}_history.pkl")
else:
    print(f"   • Check current directory for saved model files")

# Next Steps
print(f"\n🚀 Next Steps:")
print(f"   1. Copy the best model to sleepy/server/ directory")
print(f"   2. Update the FER2013 detector to use the new model")
print(f"   3. Test with real human face images using test_real_face_emotions.py")
print(f"   4. Deploy to production server")

# Usage Example
print(f"\n💡 Usage Example:")
print(f"   from tensorflow.keras.models import load_model")
if final_model_path is not None:
    print(f"   model = load_model('{final_model_path}')")
else:
    print(f"   model = load_model('your_model_file.h5')")
print(f"   # Use model for emotion detection")

# Training Status
print(f"\n📋 Training Status:")
if history is not None:
    print(f"   ✅ Training completed successfully")
    print(f"   ✅ Model saved and ready for use")
elif model is not None:
    print(f"   ⚠️ Model created but training may have failed")
    print(f"   ⚠️ Check error messages above")
else:
    print(f"   ❌ Training failed - check error messages above")
    print(f"   💡 Try running cells individually to identify issues")

print(f"\n✅ Training notebook execution completed!")
print(f"\n💡 Troubleshooting Tips:")
print(f"   • If dataset not found, check the file path in cell 3")
print(f"   • If GPU errors occur, the notebook will fallback to CPU")
print(f"   • If training fails, try reducing epochs or batch size")
print(f"   • All errors are handled gracefully with fallback options")